In [ ]:
## Connect to db handler
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
import sys
from pathlib import Path

# Add project root to path
project_root = str(Path(os.path.abspath('')).parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Set environment variables for DBPATH
os.environ['DB_PATH'] = '/lunarc/nobackup/projects/snic2020-6-41/carl/dev.db'

# Now you can import using the full module path
from scripts.sqlite_backend.db_main import EasyNerDBHandler

db = EasyNerDBHandler()


# Distribution Plots

#### Single plot

In [ ]:
# Get npmi distribution in db
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

cursor = db.conn.cursor()
cursor.execute("SELECT npmi FROM v_DIS_PNM_AGGR_ROW_FACTORY WHERE fq_doc_level > 3")
npmis = cursor.fetchall()


# Convert to numpy array
npmis = np.array(npmis).flatten()
# Plot histogram
plt.figure(figsize=(10, 6))
sns.histplot(npmis, bins=50, kde=True)
plt.title('NPMI Distribution')
plt.xlabel('NPMI')
plt.ylabel('Frequency')
plt.grid()
plt.show()




In [ ]:
# plot distribution of uniq docs
cursor.execute("SELECT uniq_docs FROM v_DIS_PNM_AGGR_ROW_FACTORY WHERE fq_doc_level > 10")
# Convert to numpy array
uniq_docs = np.array(cursor.fetchall()).flatten()
# Plot histogram
plt.figure(figsize=(10, 6))
sns.histplot(uniq_docs, bins=50, kde=True)
plt.title('Unique Documents Distribution')
plt.xlabel('Unique Documents')

#### Get percentille

In [ ]:
# Get top 70 percentile of npmi
cursor.execute("SELECT npmi FROM v_DIS_PNM_AGGR_ROW_FACTORY WHERE fq_doc_level > 2")
npmis = cursor.fetchall()
# Convert to numpy array
npmis = np.array(npmis).flatten()
# Get top 70 percentile
top_70_percentile = np.percentile(npmis, 70)
top_70_percentile
# Get top 70 percentile of uniq docs

#### Unique Documents joined with npmi plot

In [ ]:
from matplotlib import pyplot as plt

from scripts.sqlite_backend.statistics.distribution_plots import DualDistributionVisualizer

cursor = db.conn.cursor()


# Example usage:
query = "SELECT npmi, uniq_docs FROM v_DIS_PNM_AGGR_ROW_FACTORY WHERE fq_doc_level > 30"
cursor.execute(query)
data = np.array(cursor.fetchall())


if data.size == 0:
    msg = "No data retrieved from the database."
    raise ValueError(msg)

visualizer = DualDistributionVisualizer(data)
visualizer.create_dual_axis_plot() \
    .add_percentile_annotations(
        percentiles=[50, 75],
        colors='darkred',
        include_zero_pmi=True,
    ) \
    .adjust_layout(width_scale=0.7) \
    .show()

In [ ]:
import numpy as np

# Sentence count distribution - Get sentences from sentences table, group by doc_id
cursor = db.conn.cursor()
query = "SELECT doc_id, COUNT(*) FROM sentences GROUP BY doc_id"
cursor.execute(query)

# Convert to numpy array
sentence_counts = np.array(cursor.fetchall())



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

# Assuming you've already executed the query code shown in your notebook
# Extract just the counts (second column)
sentence_count_values = sentence_counts[:, 1]

# Create a DataFrame for easier analysis
sentence_distribution = pd.DataFrame(sentence_counts, columns=['doc_id', 'sentence_count'])
sentence_distribution['sentence_count'] = sentence_distribution['sentence_count'].astype(int)

# Basic statistics
stats_summary = {
    'mean': np.mean(sentence_count_values),
    'median': np.median(sentence_count_values),
    'min': np.min(sentence_count_values),
    'max': np.max(sentence_count_values),
    'std': np.std(sentence_count_values),
    'total_documents': len(sentence_count_values),
}

stats_summary


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.ticker import ScalarFormatter

# Create value counts for discrete visualization
sentence_counts_df = pd.DataFrame(sentence_count_values, columns=['sentence_count'])
count_distribution = sentence_counts_df['sentence_count'].value_counts().sort_index().reset_index()
count_distribution.columns = ['sentence_count', 'frequency']

# Create a figure with reduced width
plt.figure(figsize=(7, 6))

# Bar chart for discrete values with new color (rgb(195, 194, 140))
bar_color = (195/255, 194/255, 140/255)  # Convert RGB values to 0-1 range for matplotlib
plt.bar(count_distribution['sentence_count'], count_distribution['frequency'],
        color=bar_color, alpha=0.8, width=0.8, edgecolor='black')

# plt.title('Distribution of Sentence Counts per Document')
plt.xlabel('Number of Sentences')
plt.ylabel('Frequency (Number of Documents)')
plt.grid(alpha=0.3, axis='y')

# Set integer ticks on x-axis
max_display = 30
plt.xlim(-0.5, max_display + 0.5)
plt.xticks(np.arange(0, max_display + 1, 1))

# Configure y-axis to use regular numbers instead of scientific notation
y_formatter = ScalarFormatter(useOffset=False)
y_formatter.set_scientific(False)
plt.gca().yaxis.set_major_formatter(y_formatter)

# Add vertical lines for mean and median
plt.axvline(stats_summary['mean'], color='red', linestyle='dashed', linewidth=1,
            label=f"Mean: {stats_summary['mean']:.2f}")
plt.axvline(stats_summary['median'], color='green', linestyle='dashed', linewidth=1,
            label=f"Median: {stats_summary['median']:.2f}")
plt.legend(loc='upper left', bbox_to_anchor=(0.5, 0.7))

plt.tight_layout()
plt.show()

In [ ]:
# Word count distribution - Get word counts from sentences table, group by doc_id
cursor = db.conn.cursor()

# Query to calculate word counts per document
# This counts words by counting spaces and adding 1
query = """
SELECT doc_id, word_count FROM documents WHERE word_count > 0
"""
cursor.execute(query)
# Convert to numpy array
results = np.array(cursor.fetchall())

In [ ]:
%autoreload 2
from scripts.sqlite_backend.statistics.histograms_for_doc_distribution import (
    DocumentMetricsVisualizer,
)

# Apply the visualization to word count data with enhanced x-axis reference lines
word_count_visualizer = DocumentMetricsVisualizer(results, "word")

# Create histogram with additional reference lines
word_count_visualizer.plot_histogram(
    color=(197/255, 191/255, 191/255),
    bins=300,
    x_min=0,
    x_max=600,
    show_stats=True,
    show_std_dev=True,
    # x_ticks_count=15
    # show_quartiles=True,  # Show Q1 and Q3 lines
    # show_percentiles=[10, 90]  # Show 10th and 90th percentile lines
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.ticker import ScalarFormatter

# Create value counts for discrete visualization
word_counts_df = pd.DataFrame(sentence_count_values, columns=['word_count'])
word_counts_distribution = word_counts_df['word_count'].value_counts().sort_index().reset_index()
word_counts_distribution.columns = ['word_count', 'frequency']

# Create a figure with reduced width
plt.figure(figsize=(7, 6))

# Bar chart for discrete values with new color (rgb(195, 194, 140))
bar_color = (195/255, 194/255, 140/255)  # Convert RGB values to 0-1 range for matplotlib
plt.bar(word_counts_distribution['word_count'], word_counts_distribution['frequency'],
        color=bar_color, alpha=0.8, width=0.8, edgecolor='black')

plt.title('Distribution of Word Counts per Document')  # Updated title to reflect word counts
plt.xlabel('Number of Words')
plt.ylabel('Frequency (Number of Documents)')
plt.grid(alpha=0.3, axis='y')

# Set integer ticks on x-axis
max_display = 30
plt.xlim(-0.5, max_display + 0.5)
plt.xticks(np.arange(0, max_display + 1, 1))

# Configure y-axis to use regular numbers instead of scientific notation
y_formatter = ScalarFormatter(useOffset=False)
y_formatter.set_scientific(False)
plt.gca().yaxis.set_major_formatter(y_formatter)

# Add vertical lines for mean and median
plt.axvline(stats_summary['mean'], color='red', linestyle='dashed', linewidth=1,
            label=f"Mean: {stats_summary['mean']:.2f}")
plt.axvline(stats_summary['median'], color='green', linestyle='dashed', linewidth=1,
            label=f"Median: {stats_summary['median']:.2f}")
plt.legend(loc='upper left', bbox_to_anchor=(0.5, 0.7))

plt.tight_layout()
plt.show()

In [ ]:
# Extract just the counts (second column)
word_count_values = word_counts[:, 0]

In [ ]:
print(f"Shape of word_counts array: {word_counts.shape}")
# Flatten the array
word_counts = word_counts.flatten()
print(f"Shape of word_counts array after flattening: {word_counts.shape}")

In [ ]:
from typing import Dict, Optional, Tuple, Union

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import ScalarFormatter


class DocumentMetricVisualizer:
    """Class for visualizing document-level metric distributions."""

    def __init__(self, metric_data: np.ndarray, metric_name: str):
        """Initialize with document metric data.

        Args:
            metric_data: Array of metric values (can be 1D or 2D)
            metric_name: Name of the metric (e.g., "word", "sentence")

        """
        self.metric_name = metric_name
        self.metric_values = self._extract_values(metric_data)
        self.stats_summary = self._calculate_statistics()

    def _extract_values(self, data_array: np.ndarray) -> np.ndarray:
        """Extract metric values regardless of array dimension."""
        # First extract the values based on array shape
        if len(data_array.shape) > 1 and data_array.shape[1] == 1:
            values = data_array.flatten()
        elif len(data_array.shape) > 1 and data_array.shape[1] > 1:
            values = data_array[:, 1]
        else:
            values = data_array

        # Handle None values - convert to pandas Series for easier handling
        values_series = pd.Series(values)

        # Filter out None/NaN values and convert to integers
        clean_values = values_series.dropna().astype(int).values

        # Report if any values were removed
        dropped_count = len(values) - len(clean_values)
        if dropped_count > 0:
            print(f"Warning: Removed {dropped_count} null values from the dataset")

        return clean_values

    def _calculate_statistics(self) -> dict[str, float]:
        """Calculate descriptive statistics for the metric values."""
        return {
            'mean': np.mean(self.metric_values),
            'median': np.median(self.metric_values),
            'min': np.min(self.metric_values),
            'max': np.max(self.metric_values),
            'std': np.std(self.metric_values),
            'variance': np.var(self.metric_values),
            'total_documents': len(self.metric_values),
        }

    def print_statistics(self) -> None:
        """Print the calculated statistics."""
        for stat_name, stat_value in self.stats_summary.items():
            print(f"{stat_name}: {stat_value}")

    def plot_distribution(self,
                          color: tuple[float, float, float] = (197/255, 191/255, 191/255),
                          figsize: tuple[int, int] = (7, 6),
                          max_display: Optional[int] = None,
                          legend_pos: str = 'best',
                          save_path: Optional[str] = None) -> None:
        """Plot the distribution of metric values.

        Args:
            color: RGB color tuple (values between 0-1)
            figsize: Figure dimensions (width, height)
            max_display: Maximum value to display on x-axis (defaults to 95th percentile)
            legend_pos: Position of the legend ('best', 'upper right', etc.)
            save_path: Path to save the figure (None for no saving)

        """
        # Create value counts for discrete visualization
        metric_df = pd.DataFrame(self.metric_values, columns=[f'{self.metric_name}_count'])
        count_distribution = metric_df[f'{self.metric_name}_count'].value_counts().sort_index().reset_index()
        count_distribution.columns = [f'{self.metric_name}_count', 'frequency']

        # Create visualization
        fig = plt.figure(figsize=figsize)

        # Bar chart with specified color
        plt.bar(count_distribution[f'{self.metric_name}_count'],
                count_distribution['frequency'],
                color=color, alpha=0.8, width=0.8, edgecolor='black')

        plt.xlabel(f'Number of {self.metric_name.title()}s')
        plt.ylabel('Frequency (Number of Documents)')
        plt.grid(alpha=0.3, axis='y')

        # Set x-axis limits - adaptively based on data
        if max_display is None:
            max_display = min(100, np.percentile(self.metric_values, 95))
        plt.xlim(-0.5, max_display + 0.5)

        # Configure y-axis to use regular numbers instead of scientific notation
        y_formatter = ScalarFormatter(useOffset=False)
        y_formatter.set_scientific(False)
        plt.gca().yaxis.set_major_formatter(y_formatter)

        # Add mean and median reference lines
        plt.axvline(self.stats_summary['mean'], color='red', linestyle='dashed', linewidth=1,
                    label=f"Mean: {self.stats_summary['mean']:.2f}")
        plt.axvline(self.stats_summary['median'], color='green', linestyle='dashed', linewidth=1,
                    label=f"Median: {self.stats_summary['median']:.2f}")
        plt.legend(loc=legend_pos)

        plt.tight_layout()

        # Save the figure if a path is provided
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"Figure saved to {save_path}")

        plt.show()
        return fig

    def plot_boxplot(self, figsize: tuple[int, int] = (7, 6),
                     save_path: Optional[str] = None) -> None:
        """Create a boxplot of the metric values.

        Args:
            figsize: Figure dimensions (width, height)
            save_path: Path to save the figure (None for no saving)

        """
        fig = plt.figure(figsize=figsize)
        plt.boxplot(self.metric_values, vert=False, patch_artist=True)
        plt.xlabel(f'Number of {self.metric_name.title()}s')
        plt.title(f'Distribution of {self.metric_name.title()} Counts')
        plt.grid(alpha=0.3)
        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"Boxplot saved to {save_path}")

        plt.show()
        return fig


# Usage example for word counts
word_visualizer = DocumentMetricVisualizer(word_counts, "word")
word_visualizer.print_statistics()
word_visualizer.plot_distribution()

# Optional: Generate a boxplot and save to file
# word_visualizer.plot_boxplot(save_path="word_count_boxplot.png")